In [0]:
%sql
-- CUSTOMERS SILVER TABLE
CREATE OR REPLACE TABLE retail_lakehouse.silver.customers
USING DELTA
AS
SELECT DISTINCT
    CAST(CustomerID AS INT) AS CustomerID,
    INITCAP(TRIM(CustomerName)) AS CustomerName,
    LOWER(TRIM(Email)) AS Email,
    TRIM(City) AS City,
    TRIM(Address) AS Address,
    TO_DATE(LastUpdated, 'dd-MM-yyyy') AS LastUpdated
FROM retail_lakehouse.bronze.customers
WHERE CustomerID IS NOT NULL;

In [0]:
%sql
-- PRODUCTS SILVER TABLE
CREATE OR REPLACE TABLE retail_lakehouse.silver.products
USING DELTA
AS
SELECT
    CAST(ProductID AS INT) AS ProductID,
    TRIM(ProductName) AS ProductName,
    TRIM(Category) AS Category,
    CAST(UnitPrice AS DOUBLE) AS UnitPrice
FROM retail_lakehouse.bronze.products
WHERE TRY_CAST(UnitPrice AS DOUBLE) > 0;

In [0]:
%sql
-- STORES SILVER TABLE
CREATE OR REPLACE TABLE retail_lakehouse.silver.stores
USING DELTA
AS
SELECT
    CAST(StoreID AS INT) AS StoreID,
    INITCAP(TRIM(StoreName)) AS StoreName,
    TRIM(Region) AS Region
FROM retail_lakehouse.bronze.stores
WHERE Region IS NOT NULL;

In [0]:
%sql
-- SALES SILVER TABLE
CREATE OR REPLACE TABLE retail_lakehouse.silver.sales
USING DELTA
AS
SELECT DISTINCT
    CAST(TransactionID AS INT) AS TransactionID,
    CAST(CustomerID AS INT) AS CustomerID,
    CAST(ProductID AS INT) AS ProductID,
    CAST(StoreID AS INT) AS StoreID,
    CAST(Quantity AS INT) AS Quantity,
    TO_DATE(TxnDate, 'dd-MM-yyyy') AS TxnDate
FROM retail_lakehouse.bronze.sales
WHERE TRY_CAST(Quantity AS INT) > 0;

In [0]:
%sql
-- ENABLE CDC (CHANGE DATA FEED)
-- IMPORTANT:
-- Must happen immediately after table creation
ALTER TABLE retail_lakehouse.silver.customers
SET TBLPROPERTIES (
    delta.enableChangeDataFeed = true
);
ALTER TABLE retail_lakehouse.silver.sales
SET TBLPROPERTIES (
    delta.enableChangeDataFeed = true
);

In [0]:
%sql
-- SILVER LAYER VALIDATION
SELECT COUNT(*) AS customer_count
FROM retail_lakehouse.silver.customers;

SELECT COUNT(*) AS product_count
FROM retail_lakehouse.silver.products;

SELECT COUNT(*) AS store_count
FROM retail_lakehouse.silver.stores;

SELECT COUNT(*) AS sales_count
FROM retail_lakehouse.silver.sales;

In [0]:
%sql
-- INVALID DATA VALIDATION
SELECT *
FROM retail_lakehouse.silver.sales
WHERE Quantity <= 0;

SELECT *
FROM retail_lakehouse.silver.products
WHERE UnitPrice <= 0;

In [0]:
%sql
-- NULL VALIDATION
SELECT *
FROM retail_lakehouse.silver.sales
WHERE TransactionID IS NULL;

SELECT *
FROM retail_lakehouse.silver.customers
WHERE CustomerID IS NULL;

In [0]:
%sql
-- DUPLICATE VALIDATION
SELECT
    TransactionID,
    COUNT(*)
FROM retail_lakehouse.silver.sales
GROUP BY TransactionID
HAVING COUNT(*) > 1;

In [0]:
%sql
-- TRANSFORMATION VALIDATION
-- Proper Case Validation

SELECT CustomerName
FROM retail_lakehouse.silver.customers
WHERE CustomerName != INITCAP(CustomerName);

-- Lowercase Validation

SELECT Email
FROM retail_lakehouse.silver.customers
WHERE Email != LOWER(Email);

-- Date Validation

SELECT TxnDate
FROM retail_lakehouse.silver.sales
LIMIT 10;

In [0]:
%sql
-- VERIFY CDC ENABLEMENT
DESCRIBE HISTORY retail_lakehouse.silver.sales;
DESCRIBE HISTORY retail_lakehouse.silver.customers;